In [1]:
!wget https://www.dropbox.com/s/6zp3pwwkem2qgnz/thairath-222_final.csv.zip?dl=0 https://www.dropbox.com/s/mh4ujh7clszcddc/thaisum.csv.zip?dl=0

--2023-04-29 08:55:12--  https://www.dropbox.com/s/6zp3pwwkem2qgnz/thairath-222_final.csv.zip?dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /s/raw/6zp3pwwkem2qgnz/thairath-222_final.csv.zip [following]
--2023-04-29 08:55:12--  https://www.dropbox.com/s/raw/6zp3pwwkem2qgnz/thairath-222_final.csv.zip
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc43c8d0e87ccd6fd0933445bb3a.dl.dropboxusercontent.com/cd/0/inline/B7G9k4qbL2Pyfs6WtfxUtCJyLcPh9RZTCxvew1nx8Hu4DL5zhk31JpOq90bCEz8UpfBpSRYg6ZYKGSyOELKZYFU4YFnkkxVaS-5HuJtnIF8WBsKOzw9J19KejSP_rQ7hDGJlglMAbQdQzuZiFD1ydDq4ivMhPfOutWlIcCme3Tw-Tg/file# [following]
--2023-04-29 08:55:12--  https://uc43c8d0e87ccd6fd0933445bb3a.dl.dropboxusercontent.com/cd/0/inline/B7G9k4qbL2Pyfs6WtfxUtCJyLcPh9RZTC

In [ ]:
!unzip /content/thairath-222_final.csv.zip?dl=0

Archive:  /content/thairath-222_final.csv.zip?dl=0
  inflating: thairath-222_final.csv  
  inflating: __MACOSX/._thairath-222_final.csv  


In [ ]:
import pandas as pd
df = pd.read_csv('/content/thairath-222_final.csv')

# PROJ: mT5

In [ ]:
!nvidia-smi

Thu Apr 27 10:43:29 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.85.12    Driver Version: 525.85.12    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   39C    P8    10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install adapter-transformers datasets evaluate rouge_score sentencepiece


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 66.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.7/468.7 kB 48.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 70.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 28.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
tokenizer = T5Tokenizer.from_pretrained("thanathorn/mt5-cpe-kmutt-thai-sentence-sum")

In [ ]:
df['body'] = df['body'].apply(lambda x: x[:500])

We try to make ds dict that has keys consisting of 'input_ids', 'attention_mask', and 'labels'.

ds['input_ids'] stores tokenized indices of Thairath news body (max char=500)

ds['attention_mask'] stores attention masks of tokenized news body

ds['labels'] stores tokenized indices of Thairath news summary

In [ ]:
ds = tokenizer(list(df['body']),truncation = True,padding=True)
ds['labels'] = tokenizer(list(df['summary']), truncation = True, padding = True).input_ids

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
model = T5ForConditionalGeneration.from_pretrained('thanathorn/mt5-cpe-kmutt-thai-sentence-sum')

You are using a model of type mt5 to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


Save pickle of ds dict for reproducing the code

In [ ]:
import pickle
with open('tokenized_data.pkl', 'wb') as f:
    pickle.dump(ds, f)

In [ ]:
!zip -r tokenized_dataset.zip tokenized_data.pkl

  adding: tokenized_data.pkl (deflated 84%)


In [ ]:
cp /content/tokenized_dataset.zip /content/drive/MyDrive

ds dict is available for downloading: https://drive.google.com/file/d/1uR1FjtOpWUqxa0YcbRGGUiTy7m8LToIs/view?usp=share_link

In [ ]:
!cp /content/drive/MyDrive/tokenized_dataset.zip /content/ 

In [ ]:
!unzip /content/tokenized_dataset.zip

Archive:  /content/tokenized_dataset.zip
replace tokenized_data.pkl? [y]es, [n]o, [A]ll, [N]one, [r]ename: ์์N
error:  invalid response [์์N]
replace tokenized_data.pkl? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
import pickle
with open('/content/tokenized_data.pkl', 'rb') as f:
    ds = pickle.load(f)

For training the T5 model, split train eval test and also create datasets.Dataset from ds dict

In [ ]:
train_num=int(0.6*len(ds['input_ids']))
eval_num = int((len(ds['input_ids'])-train_num)/2)
test_num = len(ds['input_ids'])-(train_num+eval_num)
print(train_num,eval_num,test_num)
print(train_num+eval_num+test_num)

133531 44511 44511
222553


In [ ]:
len(ds['input_ids'])

222553

In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_dict({'input_ids':ds['input_ids'][:train_num],'attention_mask':ds['attention_mask'][:train_num],'labels':ds['labels'][:train_num]})
eval_dataset = Dataset.from_dict({'input_ids':ds['input_ids'][train_num:train_num+eval_num],'attention_mask':ds['attention_mask'][train_num:train_num+eval_num],'labels':ds['labels'][train_num:train_num+eval_num]})
test_dataset = Dataset.from_dict({'input_ids':ds['input_ids'][train_num+eval_num:],'attention_mask':ds['attention_mask'][train_num+eval_num:],'labels':ds['labels'][train_num+eval_num:]})

In [ ]:
del ds

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

In [ ]:
import numpy as np
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

Instead of full fine-tuning mT5 model for our Thairath text summarization, add MAM adapter and train only MAM adapter that is attached to mT5 model. This parameter-efficient fine-tuning process only train few parameters of adapter compared to training all mT5 model parameters.

For more information about MAM adapter: https://openreview.net/forum?id=0RDcd5Axok 

In [ ]:
from transformers.adapters import MAMConfig
config = MAMConfig()
model.add_adapter("mam_adapter", config=config)
# adapter_name = model.load_adapter("mam_adapter", config-config, source="ah")


In [ ]:
model.train_adapter("mam_adapter")

from transformers import TrainingArguments, AdapterTrainer
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer,model='thanathorn/mt5-cpe-kmutt-thai-sentence-sum')
training_args = TrainingArguments(   
    output_dir = './mam-mt5',       # output directory
    num_train_epochs=3,              # total # of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
                # directory for storing logs
)
trainer = AdapterTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset = eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,

)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


In [ ]:
train_results=trainer.train()

/usr/local/lib/python3.9/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 133531
  Num Epochs = 3
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 25038
  Number of trainable parameters = 60390240


Step,Training Loss
500,2.341000
1000,1.471000
1500,1.285700
2000,1.236200
2500,1.201500
3000,1.174900
3500,1.164500
4000,1.137900
4500,1.134100
5000,1.126700


Saving model checkpoint to ./mam-mt5/checkpoint-500
Configuration saved in ./mam-mt5/checkpoint-500/mam_adapter/adapter_config.json
Module weights saved in ./mam-mt5/checkpoint-500/mam_adapter/pytorch_adapter.bin
Configuration saved in ./mam-mt5/checkpoint-500/mam_adapter/head_config.json
Module weights saved in ./mam-mt5/checkpoint-500/mam_adapter/pytorch_model_head.bin
Saving model checkpoint to ./mam-mt5/checkpoint-1000
Configuration saved in ./mam-mt5/checkpoint-1000/mam_adapter/adapter_config.json
Module weights saved in ./mam-mt5/checkpoint-1000/mam_adapter/pytorch_adapter.bin
Configuration saved in ./mam-mt5/checkpoint-1000/mam_adapter/head_config.json
Module weights saved in ./mam-mt5/checkpoint-1000/mam_adapter/pytorch_model_head.bin
Saving model checkpoint to ./mam-mt5/checkpoint-1500
Configuration saved in ./mam-mt5/checkpoint-1500/mam_adapter/adapter_config.json
Module weights saved in ./mam-mt5/checkpoint-1500/mam_adapter/pytorch_adapter.bin
Configuration saved in ./mam-mt

In [ ]:
trainer.model.save_pretrained('/content/my-mam-mt5')

Configuration saved in /content/my-mam-mt5/config.json
Configuration saved in /content/my-mam-mt5/generation_config.json
The model is bigger than the maximum size per checkpoint (10GB) and is going to be split in 2 checkpoint shards. You can find where each parameters has been saved in the index located at /content/my-mam-mt5/pytorch_model.bin.index.json.


In [ ]:
model.save_adapter('/content/my-mam-mt5', 'mam_adapter')


Configuration saved in /content/my-mam-mt5/adapter_config.json
Module weights saved in /content/my-mam-mt5/pytorch_adapter.bin
Configuration saved in /content/my-mam-mt5/head_config.json
Module weights saved in /content/my-mam-mt5/pytorch_model_head.bin


In [ ]:
!zip /content/my-mam-mt5.zip /content/my-mam-mt5/*

  adding: content/my-mam-mt5/adapter_config.json (deflated 66%)
  adding: content/my-mam-mt5/config.json (deflated 66%)
  adding: content/my-mam-mt5/generation_config.json (deflated 29%)
  adding: content/my-mam-mt5/head_config.json (deflated 35%)
  adding: content/my-mam-mt5/pytorch_adapter.bin (deflated 8%)
  adding: content/my-mam-mt5/pytorch_model-00001-of-00002.bin (deflated 26%)
  adding: content/my-mam-mt5/pytorch_model-00002-of-00002.bin (deflated 7%)
  adding: content/my-mam-mt5/pytorch_model.bin.index.json (deflated 97%)
  adding: content/my-mam-mt5/pytorch_model_head.bin (deflated 7%)


In [ ]:
train_results

TrainOutput(global_step=25038, training_loss=1.0777219838918575, metrics={'train_runtime': 13154.6355, 'train_samples_per_second': 30.453, 'train_steps_per_second': 1.903, 'total_flos': 3.325205538559705e+17, 'train_loss': 1.0777219838918575, 'epoch': 3.0})

Here is our fine-tuning mT5 model and adapter for Thairath text summarization: https://drive.google.com/file/d/1_6O5ROqo-yRKDXWOz6z1sv1dRlSHVjY7/view?usp=share_link 

In [ ]:
cp /content/my-mam-mt5.zip /content/drive/MyDrive

##Rouge Score of test_dataset

Download test predicted summary dataset from our mam-mT5 model: 

In [ ]:
!gdown https://drive.google.com/u/0/uc?id=1aQjntPB3UtPywyzWYpd-I5Aq-OBIw0-P

Downloading...
From: https://drive.google.com/u/0/uc?id=1aQjntPB3UtPywyzWYpd-I5Aq-OBIw0-P
To: /content/df_pred.csv
100% 22.0M/22.0M [00:00<00:00, 67.0MB/s]


In [ ]:
import pandas as pd
test_df = pd.read_csv('./df_pred.csv')

In [ ]:
!pip install pythainlp

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 68.6 MB/s eta 0:00:00


In [ ]:
!pip install attacut

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 kB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 12.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.5/993.5 kB 65.6 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13721 sha256=40f88d2fdbffad6d4578f10183aab2426136a8af49d7964f1531a6aa979a275a
  Stored in directory: /root/.cache/pip/wheels/fc/ab/d4/5da2067ac95b36618c629a5f93f809425700506f72c9732fac
  Created wheel for fire: filename=fire-0.5.0-py2.py3-none-any.whl size=116952 sha256=4912b06b2bfc7b8c470df9c186205d8b93925f1b4846a718231c2682c734670c
  Stored in directory: /root/.cache/pip/wheels/90/d4/f7/9404e5db0116bd4d43e5666eaa3e7

Use attacut as tokenizer for comparing rouge score results to the baseline.

Tokenizer of mT5 is used for model prediction only as the model is based on sentence-piece. However, calculating rouge scores requires word grams, not subword or subcharacter gram from the model tokenizer

In [ ]:
import evaluate
from pythainlp.tokenize import word_tokenize
from tqdm import tqdm
rouge = evaluate.load('rouge')
result = rouge.compute(predictions=list(test_df['test_pred']),references=list(df['summary'].iloc[-test_df.shape[0]:]),tokenizer=lambda x:word_tokenize(x,engine='attacut'))

In [ ]:
result

{'rouge1': 0.4144537243063865,
 'rouge2': 0.21415091577847406,
 'rougeL': 0.35420755583691793,
 'rougeLsum': 0.3545750358303806}